# Analysis: Extra

The code used to reason and create extra statistics

In [2]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Swapped over to polars for the big datasets
import polars as pl

cfg = pl.Config()
cfg.set_tbl_rows(100)
cfg.set_tbl_width_chars(-1)

polars.config.Config

### World Population (by age) from the UN
Estimates from [*World Population Prospects 2024*](https://population.un.org/wpp/downloads?folder=Standard%20Projections&group=Population).

In [ ]:
# Get the latest estimates from 2023
un = pl.read_csv("./extra/world_pop_un.csv")
latest_pop = un.filter(un["year"] == 2023)

# Sum all of the pop_* columns from pop_0 to pop_25 (last col of 25 is excluded)
under25s_cols = slice(un.columns.index("pop_0"), un.columns.index("pop_25"))
under_25s = (
    latest_pop["area", "iso3"]
        .with_columns(under_25s=latest_pop[un.columns[under25s_cols]].sum_horizontal())
        .group_by("iso3")
        .agg(area=pl.col("area").first(), under_25s=pl.col("under_25s").sum())
        .sort("under_25s", descending=True)
)


print("Total:", under_25s["under_25s"].sum(), "vs. Top 20:", under_25s[:20]["under_25s"].sum(), "vs.", "Others:", under_25s[20:]["under_25s"].sum())

print(under_25s[:10])

Total: 3138065 vs. Top 20: 2181816 vs. Others: 956249
shape: (10, 3)
┌──────┬─────────────────────────────────┬───────────┐
│ iso3 ┆ area                            ┆ under_25s │
│ ---  ┆ ---                             ┆ ---       │
│ str  ┆ str                             ┆ i64       │
╞══════╪═════════════════════════════════╪═══════════╡
│ IND  ┆ India                           ┆ 618214    │
│ CHN  ┆ China                           ┆ 397447    │
│ NGA  ┆ Nigeria                         ┆ 140861    │
│ PAK  ┆ Pakistan                        ┆ 140747    │
│ IDN  ┆ Indonesia                       ┆ 115138    │
│ USA  ┆ United States of America        ┆ 105433    │
│ BGD  ┆ Bangladesh                      ┆ 81670     │
│ BRA  ┆ Brazil                          ┆ 73201     │
│ COD  ┆ Democratic Republic of the Con… ┆ 68911     │
│ EGY  ┆ Egypt                           ┆ 57215     │
└──────┴─────────────────────────────────┴───────────┘


In [29]:
countries_covered = ['KOR', 'CHE', 'COL', 'ARG', 'PER', 'IND', 'TWN', 'AUT', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'EST', 'FIN', 'GRC', 'HRV', 'HUN', 'IRL', 'LTU', 'LUX', 'LVA', 'MLT', 'POL', 'PRT', 'SWE', 'SVN', 'SVK', 'IRN', 'NOR', 'VNM', 'BEL', 'ESP', 'ROU', 'TUR', 'ITA', 'FRA', 'CAN', 'AUS', 'CHN', 'NLD', 'GBR', 'USA']
world_under25_pop_total = under_25s["under_25s"].sum()

covered = under_25s.filter(under_25s["iso3"].is_in(countries_covered))
not_covered = under_25s.filter(~under_25s["iso3"].is_in(countries_covered))
world_under25_pop_covered = covered["under_25s"].sum()
print(f"World population covered by our studies: {world_under25_pop_covered} ({float(world_under25_pop_covered / world_under25_pop_total * 100):.3}%)")

not_covered[:5]

World population covered by our studies: 1445135 (46.1%)


iso3,area,under_25s
str,str,i64
"""NGA""","""Nigeria""",140861
"""PAK""","""Pakistan""",140747
"""IDN""","""Indonesia""",115138
"""BGD""","""Bangladesh""",81670
"""BRA""","""Brazil""",73201


In [ ]:
# Parse data from multiple ITU stats series.
youth_internet = (
    pl.read_csv("extra/itu-individuals-using-the-internet.csv")
        .pivot(on="seriesCode", values="dataValue")
        .rename({"i99H": "total", "HH%U4212_HHC15to24": "15_to_24", "HH%U4212_HHC25to74": "25_to_74", "HH%U4212_HHCLess15": "under_15", "entityIso": "iso3"})
        ["iso3", "entityName", "total", "under_15", "15_to_24"]
        .group_by("iso3")
        .agg(
            pl.col("entityName").first(), 
            pl.col("total").first(ignore_nulls=True), 
            pl.col("under_15").first(ignore_nulls=True), 
            pl.col("15_to_24").first(ignore_nulls=True),
        )
        .rename({"under_15": "usage_under_15", "15_to_24": "usage_15_to_24", "total": "usage_overall"})
)

# Population data, for weighting.
under_15_cols = slice(un.columns.index("pop_0"), un.columns.index("pop_15"))
f15_to_24s_cols = slice(un.columns.index("pop_15"), un.columns.index("pop_25"))

under_15s =  (
    latest_pop["area", "iso3"]
        .with_columns(under_15s=latest_pop[un.columns[under_15_cols]].sum_horizontal())
        .group_by("iso3")
        .agg(area=pl.col("area").first(), under_15s=pl.col("under_15s").sum())
)
f15_to_24s =  (
    latest_pop["area", "iso3"]
        .with_columns(f15_to_24s=latest_pop[un.columns[f15_to_24s_cols]].sum_horizontal())
        .group_by("iso3")
        .agg(area=pl.col("area").first(), f15_to_24s=pl.col("f15_to_24s").sum())
        
)
youth_pop = (
    under_15s
        .join(f15_to_24s, on="iso3", how="left")
        .rename({"under_15s": "pop_under_15", "f15_to_24s": "pop_15_to_24"})
        ["iso3", "area", "pop_under_15", "pop_15_to_24"]
)
youth_pop = youth_pop.with_columns(pop_youth=youth_pop["pop_under_15"] + youth_pop["pop_15_to_24"])

# Combining the two sources.
youth_internet_usage = (
    youth_internet.join(youth_pop, on="iso3", how="left")
    ["iso3", "entityName", "pop_youth", "pop_under_15", "pop_15_to_24", "usage_overall", "usage_under_15", "usage_15_to_24"]
    .filter(pl.col("pop_youth") > 0)
)

all_youth_data_available = (
    youth_internet_usage
        .filter(pl.all_horizontal("usage_under_15", "usage_15_to_24").is_not_null())
        .with_columns(
            youth_usage_index=(
                pl.col("pop_under_15") * pl.col("usage_under_15") / 100 + 
                pl.col("pop_15_to_24") * pl.col("usage_15_to_24") / 100
            ),
            method=pl.lit("WW")
        )
)

f14t25_available = (
    youth_internet_usage
        .filter(pl.col("usage_15_to_24").is_not_null().and_(pl.col("usage_under_15").is_null()))
        .with_columns(
            youth_usage_index=(pl.col("pop_youth") * pl.col("usage_15_to_24") / 100),
            method=pl.lit("EW")
        )
)

u15_available = (
    youth_internet_usage
        .filter(pl.col("usage_15_to_24").is_null().and_(pl.col("usage_under_15").is_not_null()))
        .with_columns(
            youth_usage_index=(pl.col("pop_youth") * pl.col("usage_under_15") / 100),
            method=pl.lit("EW")
        )
)

no_youth_data_available = (
    youth_internet_usage
        .filter(pl.col("usage_15_to_24").is_null().and_(pl.col("usage_under_15").is_null()))
        .with_columns(
            youth_usage_index=(pl.col("pop_youth") * pl.col("usage_overall") / 100),
            method=pl.lit("EE")
        )
)

youth_internet_index = (
    pl.concat((
        all_youth_data_available,
        f14t25_available,
        u15_available,
        no_youth_data_available
    ))
    .sort("youth_usage_index", descending=True)
)

(
    youth_internet_index[:10]
    # .filter(pl.col("iso3").is_in(countries_covered).not_())
)

iso3,entityName,pop_youth,pop_under_15,pop_15_to_24,usage_overall,usage_under_15,usage_15_to_24,youth_usage_index,method
str,str,i64,i64,i64,f64,f64,f64,f64,str
"""IND""","""India""",618214,360338,257876,60.2523,null,null,372488.153922,"""EE"""
"""CHN""","""China""",397447,236001,161446,90.6,null,null,360086.982,"""EE"""
"""USA""","""United States""",105433,60432,45001,93.5256,null,null,98606.845848,"""EE"""
"""IDN""","""Indonesia""",115138,70113,45025,69.2084,63.8064,94.3128,87200.919432,"""WW"""
"""PAK""","""Pakistan""",140747,91669,49078,57.253,null,null,80581.87991,"""EE"""
"""BRA""","""Brazil""",73201,42094,31107,84.1506,90.7697,95.211,67825.883288,"""WW"""
"""NGA""","""Nigeria""",140861,94553,46308,40.064,null,null,56434.55104,"""EE"""
"""MEX""","""Mexico""",54517,32329,22188,81.1832,77.6567,96.3225,46477.670843,"""WW"""
"""EGY""","""Egypt""",57215,37149,20066,74.0119,null,null,42345.908585,"""EE"""
